In [1]:
!apt-get update -y && apt-get install -y iverilog
!pip -q install requests

import os, re, json, textwrap, subprocess
import requests

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,916 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,613 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:

In [2]:
SPEC = """
Design a synthesizable Verilog module named binary_to_bcd_converter with:
- input  [15:0] binary_input
- output [19:0] bcd_output

bcd_output format: {ten_thousands[3:0], thousands[3:0], hundreds[3:0], tens[3:0], ones[3:0]}
For binary_input 0..65535, convert to decimal and output the 5 BCD digits.
Purely combinational (no clocks).
Use a synthesizable double-dabble (shift-add-3) implementation.
"""

PROMPT = f"""
You are an expert digital design engineer.

Task: Write synthesizable Verilog code that compiles with:
iverilog -g2012

CRITICAL OUTPUT RULES (you MUST follow these exactly):
1) Your ENTIRE response must be ONLY a single fenced code block: ```verilog ... ```
2) NO explanations, NO prose, NO comments outside the code block.
3) NO manual step-by-step solving. Output CODE only.
4) The code block must contain ONLY the module — no testbench.
5) Do NOT include a `timescale directive.
6) You MUST start your module with EXACTLY this header, copy it verbatim:

module binary_to_bcd_converter(
    input  [15:0] binary_input,
    output [19:0] bcd_output
);

CRITICAL IMPLEMENTATION RULES:
7) Use a single reg [19:0] shift_reg and integer loop variable i.
8) Use ONE always @* block containing a for loop that runs exactly 16 times (i=0 to 15).
9) Loop body structure MUST be exactly:
       if (shift_reg[19:16] > 4) shift_reg[19:16] = shift_reg[19:16] + 3;
       if (shift_reg[15:12] > 4) shift_reg[15:12] = shift_reg[15:12] + 3;
       if (shift_reg[11:8]  > 4) shift_reg[11:8]  = shift_reg[11:8]  + 3;
       if (shift_reg[7:4]   > 4) shift_reg[7:4]   = shift_reg[7:4]   + 3;
       if (shift_reg[3:0]   > 4) shift_reg[3:0]   = shift_reg[3:0]   + 3;
       shift_reg = {{shift_reg[18:0], binary_input[15-i]}};
10) Use plain integers for comparisons and additions (> 4, + 3), NOT sized literals.
11) Initialize shift_reg = 20'b0 before the loop.
12) assign bcd_output = shift_reg; after the always block.

Spec:
{SPEC}
""".strip()

print("PROMPT ready.")

PROMPT ready.


In [3]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini model
model = genai.GenerativeModel('gemini-2.5-flash')
completion = model.generate_content(PROMPT)
print(completion.text)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


```verilog
module binary_to_bcd_converter(
    input  [15:0] binary_input,
    output [19:0] bcd_output
);

    reg [19:0] shift_reg;
    integer i;

    always @* begin
        shift_reg = 20'b0; 
        for (i = 0; i < 16; i = i + 1) begin
            if (shift_reg[19:16] > 4) shift_reg[19:16] = shift_reg[19:16] + 3;
            if (shift_reg[15:12] > 4) shift_reg[15:12] = shift_reg[15:12] + 3;
            if (shift_reg[11:8]  > 4) shift_reg[11:8]  = shift_reg[11:8]  + 3;
            if (shift_reg[7:4]   > 4) shift_reg[7:4]   = shift_reg[7:4]   + 3;
            if (shift_reg[3:0]   > 4) shift_reg[3:0]   = shift_reg[3:0]   + 3;
            shift_reg = {shift_reg[18:0], binary_input[15-i]};
        end
    end

    assign bcd_output = shift_reg;

endmodule
```


In [4]:
def extract_sv_code(text: str) -> str:
    """Extract verilog code from a fenced ```verilog ... ``` block."""
    pattern = r"```verilog\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    pattern = r"```\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

sv_code = extract_sv_code(completion.text)
print("EXTRACTED VERILOG (preview)")
print(sv_code)

os.makedirs("bcd_ext", exist_ok=True)
with open("bcd_ext/binary_to_bcd_converter.v", "w") as f:
    f.write(sv_code)

print("\nSaved to bcd_ext/binary_to_bcd_converter.v")

EXTRACTED VERILOG (preview)
module binary_to_bcd_converter(
    input  [15:0] binary_input,
    output [19:0] bcd_output
);

    reg [19:0] shift_reg;
    integer i;

    always @* begin
        shift_reg = 20'b0; 
        for (i = 0; i < 16; i = i + 1) begin
            if (shift_reg[19:16] > 4) shift_reg[19:16] = shift_reg[19:16] + 3;
            if (shift_reg[15:12] > 4) shift_reg[15:12] = shift_reg[15:12] + 3;
            if (shift_reg[11:8]  > 4) shift_reg[11:8]  = shift_reg[11:8]  + 3;
            if (shift_reg[7:4]   > 4) shift_reg[7:4]   = shift_reg[7:4]   + 3;
            if (shift_reg[3:0]   > 4) shift_reg[3:0]   = shift_reg[3:0]   + 3;
            shift_reg = {shift_reg[18:0], binary_input[15-i]};
        end
    end

    assign bcd_output = shift_reg;

endmodule

Saved to bcd_ext/binary_to_bcd_converter.v


In [5]:
testbench = """
`timescale 1ns/1ps

module binary_to_bcd_converter_tb;

    reg  [15:0] binary_input;
    wire [19:0] bcd_output;

    binary_to_bcd_converter dut (
        .binary_input(binary_input),
        .bcd_output(bcd_output)
    );

    integer i;
    integer errors;
    reg [3:0] ones, tens, hundreds, thousands, ten_thousands;
    integer val;

    task check;
        input [15:0] test_val;
        integer expected_ones, expected_tens, expected_hundreds;
        integer expected_thousands, expected_ten_thousands;
        begin
            binary_input = test_val;
            #10;
            val             = test_val;
            expected_ones        = val % 10;       val = val / 10;
            expected_tens        = val % 10;       val = val / 10;
            expected_hundreds    = val % 10;       val = val / 10;
            expected_thousands   = val % 10;       val = val / 10;
            expected_ten_thousands = val % 10;

            ones          = bcd_output[3:0];
            tens          = bcd_output[7:4];
            hundreds      = bcd_output[11:8];
            thousands     = bcd_output[15:12];
            ten_thousands = bcd_output[19:16];

            if (ones          != expected_ones         ||
                tens          != expected_tens         ||
                hundreds      != expected_hundreds     ||
                thousands     != expected_thousands    ||
                ten_thousands != expected_ten_thousands) begin
                $display("FAIL: input=%0d  got=%0d%0d%0d%0d%0d  expected=%0d%0d%0d%0d%0d",
                    test_val,
                    ten_thousands, thousands, hundreds, tens, ones,
                    expected_ten_thousands, expected_thousands,
                    expected_hundreds, expected_tens, expected_ones);
                errors = errors + 1;
            end
        end
    endtask

    initial begin
        errors = 0;

        // Test boundaries
        check(16'd0);
        check(16'd1);
        check(16'd9);
        check(16'd10);
        check(16'd99);
        check(16'd100);
        check(16'd255);
        check(16'd999);
        check(16'd1000);
        check(16'd9999);
        check(16'd10000);
        check(16'd32767);
        check(16'd60000);
        check(16'd65534);
        check(16'd65535);

        // Sweep a range
        for (i = 0; i <= 65535; i = i + 100)
            check(i[15:0]);

        if (errors == 0)
            $display("PASS: all tests passed.");
        else
            $display("FAIL: %0d error(s) found.", errors);

        $finish;
    end

endmodule
"""

with open("bcd_ext/binary_to_bcd_converter_tb.v", "w") as f:
    f.write(testbench)

print("Testbench written.")

Testbench written.


In [6]:
!cd bcd_ext/ && iverilog -g2012 -o bcd.vvp binary_to_bcd_converter.v binary_to_bcd_converter_tb.v && vvp bcd.vvp

PASS: all tests passed.
